In [ ]:
import json
import shutil
import pathlib
from pathlib import Path
from itertools import islice
from historical_ner_processing.digiknihovny_utils import retrieve_digiknihovny_resources

In [ ]:
data_root = pathlib.Path("../../../mnt/cuni_lf1/")
page_xmls = list(islice((f for f in data_root.rglob("*.page_xml.zip")), 150))

In [ ]:
THRESH = 0.8

data_root = pathlib.Path("../../../mnt/")

export_root = Path("./export")
export_root.mkdir(parents=True, exist_ok=True)


def parse_library_and_doc_id(page_xml_path: Path):
    library = page_xml_path.parent.name
    name = page_xml_path.name
    suffix = ".page_xml.zip"
    document_id = name[: -len(suffix)] if name.endswith(suffix) else page_xml_path.stem
    return library, document_id


ok = 0
skipped_empty = 0
skipped_lowconf = 0
fail = 0

for px in page_xmls:
    library, document_id = parse_library_and_doc_id(Path(px))
    print("\n===", library, document_id, "===")

    try:
        page_layouts, page_names, images, image_paths, face_detections, ners = (
            retrieve_digiknihovny_resources(
                data_root,
                library,
                document_id,
                return_page_xmls=True,
                return_images=True,
                return_people_gator=True,
                return_people_gator_ners=True,
            )
        )

        if not face_detections:
            print("SKIP (no face detections)")
            skipped_empty += 1
            continue

        face_detections_thr = [
            r for r in face_detections if float(r.confidence) >= THRESH
        ]
        if not face_detections_thr:
            print(f"SKIP (no detections with confidence >= {THRESH})")
            skipped_lowconf += 1
            continue

        base = export_root / library / document_id
        out_det = base / "detections"
        out_det.mkdir(parents=True, exist_ok=True)

        with (out_det / "peoplegator_labels.jsonl").open("w", encoding="utf-8") as f:
            for r in face_detections_thr:
                f.write(json.dumps(r.model_dump(), ensure_ascii=False) + "\n")

        if ners:
            with (out_det / "peoplegator_ners.jsonl").open("w", encoding="utf-8") as f:
                for r in ners:
                    f.write(json.dumps(r.model_dump(), ensure_ascii=False) + "\n")

        pages_with_faces = sorted({Path(str(r.page)).name for r in face_detections_thr})

        name_to_path = {}
        if image_paths:
            for p in image_paths:
                p = Path(p)
                name_to_path[p.name] = p

        copied_pages = 0
        for name in pages_with_faces:
            src = name_to_path.get(name, None)
            if src is None:
                cand = data_root / library / f"{document_id}.images" / name
                src = cand if cand.exists() else None

            if src is not None and Path(src).exists():
                shutil.copy2(src, out_det / name)
                copied_pages += 1

        print(
            "OK:",
            "detections_saved=",
            len(face_detections_thr),
            "pages_with_faces=",
            len(pages_with_faces),
            "pages_copied=",
            copied_pages,
            "ners=",
            0 if not ners else len(ners),
        )
        ok += 1

    except Exception as e:
        print("FAILED:", library, document_id, "->", repr(e))
        fail += 1

print(
    "\nDONE. ok=",
    ok,
    "skipped_empty=",
    skipped_empty,
    "skipped_lowconf=",
    skipped_lowconf,
    "fail=",
    fail,
)